# Maize Yield Prediction — Kenya Rift Valley
## Uasin Gishu County, Kenya

Exploratory geospatial AI workflow combining CHIRPS rainfall, Sentinel-2 NDVI, SRTM terrain and historical maize yield. The target is county-season yield; the dataset contains nine annual observations (2015–2023).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import ee, geemap, pandas as pd, numpy as np, matplotlib.pyplot as plt, pickle, warnings
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')

EE_PROJECT='crop-yield-prediction-506111'
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    try: ee.Initialize(project=EE_PROJECT)
    except Exception: ee.Initialize()

ee.Image('USGS/SRTMGL1_003').reduceRegion(ee.Reducer.mean(),ee.Geometry.Point([35.27,0.52]),30,maxPixels=1e6).getInfo()
print('✓ Earth Engine connected')

## 2A. Study area — Uasin Gishu County
Using geoBoundaries ADM1 so the same county geometry is used throughout the Earth Engine analysis.

In [ ]:
adm1=ee.FeatureCollection('WM/geoLab/geoBoundaries/600/ADM1')
kenya_counties=adm1.filter(ee.Filter.eq('shapeGroup','KEN'))
uasin_gishu=kenya_counties.filter(ee.Filter.eq('shapeName','Uasin Gishu'))

if kenya_counties.size().getInfo()==0: raise ValueError('Kenya ADM1 features were not found.')
if uasin_gishu.size().getInfo()==0:
    print(kenya_counties.aggregate_array('shapeName').getInfo())
    raise ValueError('Uasin Gishu was not found. Check the county names above.')

aoi=uasin_gishu.geometry()
print('✓ Uasin Gishu AOI created')

In [ ]:
# 2B. Kenya + selected county labels
Map=geemap.Map(center=[0.52,37.9],zoom=6,basemap='Esri.WorldImagery')
Map.add_layer(kenya_counties.style(color='FFFFFF',fillColor='00000000',width=1),{},'Kenya County Boundaries')
Map.add_layer(uasin_gishu.style(color='FF0000',fillColor='FF000044',width=4),{},'Uasin Gishu — Study Area')
context_counties=kenya_counties.filter(ee.Filter.inList('shapeName',['Trans Nzoia','Nandi','Elgeyo-Marakwet','Kericho','Kakamega','Nakuru']))
Map.add_labels(data=context_counties,column='shapeName',font_size='10pt',font_color='white',font_family='arial',font_weight='bold')
Map.add_labels(data=uasin_gishu,column='shapeName',font_size='15pt',font_color='red',font_family='arial',font_weight='bold')
Map.add_layer(ee.FeatureCollection(kenya_counties.geometry().dissolve()).style(color='FFFF00',fillColor='00000000',width=3),{},'Kenya National Boundary')
Map.centerObject(kenya_counties,6)
Map

## 3. Historical maize yield
Verify these annual values against the exact KNBS / Ministry of Agriculture publication before publication.

In [ ]:
yield_data={2015:3.2,2016:2.8,2017:3.5,2018:3.1,2019:2.9,2020:3.4,2021:3.8,2022:3.6,2023:3.3}
yield_df=pd.DataFrame(list(yield_data.items()),columns=['season_start_year','yield_tons_per_ha'])
yield_df['season']=yield_df['season_start_year'].astype(str)+'/'+(yield_df['season_start_year']+1).astype(str)
display(yield_df)

## 4. CHIRPS seasonal rainfall
Growing season: October–March. Features: seasonal total, peak monthly rainfall and monthly rainfall variability (coefficient of variation).

In [ ]:
START_YEAR,END_YEAR=2015,2023
years=list(range(START_YEAR,END_YEAR+1))
chirps=ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(aoi)

def rainfall_mean(start_date,end_date):
    d=chirps.filterDate(start_date,end_date).sum().reduceRegion(ee.Reducer.mean(),aoi,5566,maxPixels=1e13,bestEffort=True).getInfo()
    return d.get('precipitation',np.nan)

chirps_data={}
for year in years:
    total=rainfall_mean(f'{year}-10-01',f'{year+1}-04-01')
    monthly=[]
    for month in [10,11,12]:
        start=f'{year}-{month:02d}-01'; end=f'{year+1}-01-01' if month==12 else f'{year}-{month+1:02d}-01'
        monthly.append(rainfall_mean(start,end))
    for month in [1,2,3]:
        start=f'{year+1}-{month:02d}-01'; end=f'{year+1}-04-01' if month==3 else f'{year+1}-{month+1:02d}-01'
        monthly.append(rainfall_mean(start,end))
    monthly=np.asarray(monthly,dtype=float); m=np.nanmean(monthly)
    chirps_data[year]={'rainfall_total':total,'rainfall_peak':np.nanmax(monthly),'rainfall_cv':np.nanstd(monthly)/m if m>0 else np.nan,'monthly_values':monthly}
    print(f'{year}/{year+1}: {total:.1f} mm')

## 5. Sentinel-2 peak-season NDVI
December–February mean NDVI using Sentinel-2 SR harmonized imagery and Sentinel-2 cloud-probability masking.

In [ ]:
S2_SR=ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
S2_CLOUDS=ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
MAX_CLOUD_PROBABILITY=40

def mask_clouds(img):
    p=ee.Image(img.get('cloud_mask')).select('probability')
    return img.updateMask(p.lt(MAX_CLOUD_PROBABILITY))

def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8','B4']).rename('NDVI'))

ndvi_data={}; ndvi_images={}
for year in years:
    start=f'{year}-12-01'; end=f'{year+1}-03-01'
    sr=S2_SR.filterBounds(aoi).filterDate(start,end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',60))
    clouds=S2_CLOUDS.filterBounds(aoi).filterDate(start,end)
    joined=ee.Join.saveFirst('cloud_mask').apply(primary=sr,secondary=clouds,condition=ee.Filter.equals(leftField='system:index',rightField='system:index'))
    clean=ee.ImageCollection(joined).filter(ee.Filter.notNull(['cloud_mask'])).map(mask_clouds).map(add_ndvi)
    image=clean.select('NDVI').mean().clip(aoi)
    result=image.reduceRegion(ee.Reducer.mean(),aoi,10,maxPixels=1e13,bestEffort=True).getInfo()
    ndvi_mean=result.get('NDVI',np.nan)
    ndvi_data[year]={'ndvi_mean':ndvi_mean}; ndvi_images[year]=image
    print(f'{year}/{year+1}: NDVI={ndvi_mean:.3f}')

## 6. SRTM terrain
Elevation and slope are retained for map context but excluded from the annual predictive feature set because county-average terrain is effectively constant over time.

In [ ]:
srtm=ee.Image('USGS/SRTMGL1_003')
slope_image=ee.Terrain.slope(srtm)
elevation_mean=srtm.reduceRegion(ee.Reducer.mean(),aoi,30,maxPixels=1e13,bestEffort=True).getInfo().get('elevation',np.nan)
slope_mean=slope_image.reduceRegion(ee.Reducer.mean(),aoi,30,maxPixels=1e13,bestEffort=True).getInfo().get('slope',np.nan)
print(f'Mean elevation: {elevation_mean:.1f} m'); print(f'Mean slope: {slope_mean:.2f}°')

## 7. Methodological validation note
Because there are only nine annual county-level yield observations (2015–2023), an 80/20 split would leave only two test observations. Leave-One-Out Cross-Validation (LOOCV) is therefore used. The final predictive feature set contains four time-varying predictors: seasonal rainfall, peak rainfall, rainfall variability and Sentinel-2 NDVI. Random Forest and Gradient Boosting are constrained with shallow trees and minimum leaf sizes. Performance is reported using out-of-sample R², RMSE and MAE. Results are exploratory rather than evidence of an operational forecasting system. Spatial yield prediction would require geographically distributed yield observations and independent spatial validation.

In [ ]:
# 7B. Feature engineering + temporal alignment
data=[]
for year in years:
    data.append({'season_start_year':year,'season':f'{year}/{year+1}','rainfall_total':chirps_data[year]['rainfall_total'],'rainfall_peak':chirps_data[year]['rainfall_peak'],'rainfall_cv':chirps_data[year]['rainfall_cv'],'ndvi_mean':ndvi_data[year]['ndvi_mean'],'yield_tons_per_ha':yield_data[year]})
df=pd.DataFrame(data).dropna().reset_index(drop=True)
feature_columns=['rainfall_total','rainfall_peak','rainfall_cv','ndvi_mean']; target_column='yield_tons_per_ha'
assert len(df)==len(yield_data)
assert df.season_start_year.min()==START_YEAR and df.season_start_year.max()==END_YEAR
display(df)
print('✓ Temporal alignment check passed')

In [ ]:
# 8. Exploratory analysis
display(df[feature_columns+[target_column]].describe())
plt.figure(figsize=(10,5)); plt.plot(df['season'],df[target_column],marker='o',linewidth=2); plt.xlabel('Growing Season'); plt.ylabel('Maize Yield (tons/ha)'); plt.title('Historical Maize Yield — Uasin Gishu County'); plt.grid(alpha=.3); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
display(df[feature_columns+[target_column]].corr())

In [ ]:
# 9. Machine learning + LOOCV
X=df[feature_columns].copy(); y=df[target_column].copy(); loo=LeaveOneOut()
models={'Random Forest':RandomForestRegressor(n_estimators=300,max_depth=3,min_samples_leaf=2,max_features='sqrt',random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(n_estimators=100,learning_rate=.05,max_depth=2,min_samples_leaf=2,random_state=42)}
results={}
for name,model in models.items():
    pred=cross_val_predict(model,X,y,cv=loo)
    r2=r2_score(y,pred); rmse=np.sqrt(mean_squared_error(y,pred)); mae=mean_absolute_error(y,pred)
    results[name]={'model':model,'predictions':pred,'r2':r2,'rmse':rmse,'mae':mae}
    print(f'\n{name}'); print(f'LOOCV R²   : {r2:.4f}'); print(f'LOOCV RMSE : {rmse:.4f} tons/ha'); print(f'LOOCV MAE  : {mae:.4f} tons/ha')

In [ ]:
# 10. Model selection + observed vs predicted
best_name=min(results,key=lambda n:results[n]['rmse']); best=results[best_name]; best_model=best['model']; best_predictions=best['predictions']; best_r2=best['r2']; best_rmse=best['rmse']; best_mae=best['mae']
print('='*60); print(f'SELECTED MODEL: {best_name}'); print(f'LOOCV R²: {best_r2:.4f}'); print(f'LOOCV RMSE: {best_rmse:.4f} tons/ha'); print(f'LOOCV MAE: {best_mae:.4f} tons/ha'); print('='*60)
plot_min=min(y.min(),best_predictions.min()); plot_max=max(y.max(),best_predictions.max())
plt.figure(figsize=(8,6)); plt.scatter(y,best_predictions,s=100,alpha=.8); plt.plot([plot_min,plot_max],[plot_min,plot_max],'--',linewidth=2,label='Perfect prediction'); plt.xlabel('Observed Yield (tons/ha)'); plt.ylabel('LOOCV Predicted Yield (tons/ha)'); plt.title(f'{best_name}: Observed vs Predicted\nLOOCV R² = {best_r2:.3f}'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# 11. Feature importance
best_model.fit(X,y)
feature_importance_df=pd.DataFrame({'feature':feature_columns,'importance':best_model.feature_importances_}).sort_values('importance',ascending=False).reset_index(drop=True)
display(feature_importance_df)
plt.figure(figsize=(9,5)); plt.barh(feature_importance_df.feature,feature_importance_df.importance); plt.gca().invert_yaxis(); plt.xlabel('Relative Feature Importance'); plt.title(f'{best_name}: Predictor Importance'); plt.grid(axis='x',alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# 12. Final interactive GEE map
display_year=2023
Map_final=geemap.Map(center=[.52,37.9],zoom=6,basemap='Esri.WorldImagery')
Map_final.add_layer(kenya_counties.style(color='FFFFFF',fillColor='00000000',width=1),{},'Kenya County Boundaries')
Map_final.add_layer(uasin_gishu.style(color='FF0000',fillColor='FF000044',width=4),{},'Uasin Gishu — Study Area')
Map_final.add_labels(data=context_counties,column='shapeName',font_size='10pt',font_color='white',font_family='arial',font_weight='bold')
Map_final.add_labels(data=uasin_gishu,column='shapeName',font_size='15pt',font_color='red',font_family='arial',font_weight='bold')
Map_final.add_layer(ndvi_images[display_year],{'min':.1,'max':.8,'palette':['d7191c','fdae61','ffffbf','a6d96a','1a9641']},f'{display_year}/{display_year+1} Mean NDVI')
rainfall_image=chirps.filterDate(f'{display_year}-10-01',f'{display_year+1}-04-01').sum().clip(aoi)
Map_final.add_layer(rainfall_image,{'min':300,'max':800,'palette':['ffffcc','a1dab4','41b6c4','2c7fb8','253494']},f'{display_year}/{display_year+1} Seasonal Rainfall')
Map_final.add_layer(srtm.clip(aoi),{'min':1500,'max':3000,'palette':['006400','7fff00','ffff00','ffa500','8b0000']},'SRTM Elevation')
Map_final.add_layer(ee.FeatureCollection(kenya_counties.geometry().dissolve()).style(color='FFFF00',fillColor='00000000',width=3),{},'Kenya National Boundary')
Map_final.centerObject(kenya_counties,6)
Map_final

In [ ]:
# 13. Illustrative scenarios — not operational forecasts
scenarios={'Dry Year':{'rainfall_total':400,'rainfall_peak':100,'rainfall_cv':.40,'ndvi_mean':.40},'Typical Year':{'rainfall_total':550,'rainfall_peak':120,'rainfall_cv':.35,'ndvi_mean':.48},'Wet / High-Vigor Year':{'rainfall_total':650,'rainfall_peak':140,'rainfall_cv':.30,'ndvi_mean':.55}}
out=[]
for name,values in scenarios.items(): out.append({'scenario':name,'predicted_yield_tons_per_ha':best_model.predict(pd.DataFrame([values])[feature_columns])[0]})
display(pd.DataFrame(out))

## 14. Interpretation and limitations
- N=9 annual county-level observations is a major limitation.
- A negative LOOCV R² means the current model does not outperform a simple mean-yield baseline out of sample.
- Feature importance is descriptive, not causal.
- The current model predicts county-season yield, not 10 m or field-level yield.
- Spatial yield modelling requires geographically distributed yield observations and independent spatial validation.

In [ ]:
# 15. Save model, metadata and dataset
model_path='/content/drive/My Drive/maize_yield_model.pkl'; metadata_path='/content/drive/My Drive/maize_yield_model_metadata.pkl'; dataset_path='/content/drive/My Drive/uasin_gishu_maize_yield_dataset.csv'
with open(model_path,'wb') as f: pickle.dump(best_model,f)
metadata={'region':'Uasin Gishu County, Kenya','crop':'Maize','growing_season':'October-March','years':years,'feature_columns':feature_columns,'model_type':best_name,'loocv_r2':best_r2,'loocv_rmse':best_rmse,'loocv_mae':best_mae,'observation_count':len(df),'mean_elevation_m':elevation_mean,'mean_slope_degrees':slope_mean,'sentinel_cloud_probability_threshold':MAX_CLOUD_PROBABILITY,'model_scope':'County-level exploratory model'}
with open(metadata_path,'wb') as f: pickle.dump(metadata,f)
df.to_csv(dataset_path,index=False)
print('✓ Saved model, metadata and modelling dataset')